# Finding and Analzying Growth Stocks to Support Investment Decisions for 2025

# Introduction
The aim of this project is to gather information from the internet about growth stocks, and analyse them.
**Objectives**:
- Define growth stocks
- Use a webscraper to collect information from Investopedia
- Use a web sprider to gather data about the suggested stocks
- Store the collected data in a database
- Select the 3 most frequently mentioned stocks
- Prepare data for analysis
- Analyse data
- Summarize insights

## What will the analysis be focus on?
The analysis focuses on:
- visualize the most frequent words used when mentioning the specific stocks
- analyze the context around the stock's company; is it mostly positive or negative? Probably we don't want to invest in a company which has a negative publicity
- *volatility*: check the difference between the highest and lowest values within a day. Also, examine the difference between the opening and closing prices on the same day, and compare the opening price with the previous day's closing price to see the mood changes while the market is closed.
- *monotonity*: analyze the closing prices to determine the overall trend of the security

## Growth Stocks

#### What Is a Growth Stock?
A growth stock is any share in a company that is anticipated to grow at a rate significantly above the average growth for the market$^1$

#### What are the benefits of investing in growth stocks?
These companies typically reinvest their earnings into the business to fuel further expansion, which often leads to impressive revenue and profit growth over time.

---

# SQLight Database and Class
Ceate a database to store data which will be analysed later. 

Considerations:
- Rarely, but it can happen that one company has more tickers
- When looking for tickers, it may not be possible to scrape company name and sector from the texts at first, so those are optional, can be filled with later searches
- Context is text scraped from where the company and/or sector was mentioned if exists
- Link is the URL of the website where the ccompany and/or sector was mentioned if exists

<div align="left">
    
| Tables         | Columns                                      |
|:---------------|:---------------------------------------------|
| **stock_info** | Ticker - text, primary key  |
|               | Company name - optional text  |
|               | Sector - optional text  |
|               | Investopedia_link - optional text  |
| **company_context** | Company - text, primary key  |
|               | Title - optional text  |
|               | Article - optional text  |
|               | Link - optional text  |
| **sector_context** | Sector - text, primary key  |
|               | Title - optional text  |
|               | Article - optional text  |
|               | Link - optional text  |

</div>

## Create tables

In [1]:
import sqlite3 as sql

# To avoid conflict with different versions
%config SqlMagic.style = 'Default'
%reload_ext sql

In [2]:
# %sql sqlite:///stockAnalysis.db DROP TABLE stock_info;
# %sql sqlite:///stockAnalysis.db DROP TABLE company_context;
# %sql sqlite:///stockAnalysis.db DROP TABLE sector_context;

In [3]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE IF NOT EXISTS stock_info(
    ticker VARCHAR(10) NOT NULL UNIQUE,
    company TEXT,
    sector VARCHAR(100),
    investopedia_link TEXT,
    PRIMARY KEY (ticker),
    FOREIGN KEY (company) REFERENCES company_context(company),
    FOREIGN KEY (sector) REFERENCES sector_context(sector)
);

Done.


[]

In [4]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE IF NOT EXISTS company_context(
    company TEXT NOT NULL UNIQUE,
    title TEXT,
    article TEXT,
    link TEXT, 
    PRIMARY KEY (company)
);

Done.


[]

In [5]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE  IF NOT EXISTS sector_context(
    sector VARCHAR(100) NOT NULL UNIQUE,
    title TEXT, 
    article TEXT,
    link TEXT, 
    PRIMARY KEY (sector)
);

Done.


[]

##### Show created tables

In [6]:
%%sql sqlite:///stockAnalysis.db
SELECT name
FROM sqlite_master
WHERE type='table';

Done.


name
stock_info
company_context
sector_context


## Handle Database queries

In [155]:
from io import StringIO
from contextlib import redirect_stdout
import inspect

STOCK_DB = "%sql sqlite:///stockAnalysis.db"
MOCK_DB = "%sql sqlite:///:memory:"

class StockAnalysisDB:
    debug = None
    mock_db = None

    def __init__(self, mock_db = False, debug = False):
        self.debug = debug
        self.mock_db = mock_db
        
    def validate_dict_raise(self, data):
        if not (isinstance(data, list) and all(isinstance(row, dict) for row in data)):
            raise TypeError("Data is not a list of dictionaries")

    def call_query(self, query):
        if self.debug:
            print(f"{self.__class__.__name__}.{inspect.currentframe().f_code.co_name}")
    
        # sql magic doesn't throw just writes to stdout
        ret = None
        output = StringIO()
        with redirect_stdout(output):
            if self.mock_db:
                if self.debug:
                    print(f"mock db is used")
                ret = %sql sqlite:///:memory: {query}
            else:
                if self.debug:
                    print(f"stock db is used")
                ret = %sql sqlite:///stockAnalysis.db {query}
        
        msg = output.getvalue()
        if self.debug:
            print(f"{self.__class__.__name__}.{inspect.currentframe().f_code.co_name}")
        
        if 'sqlite3.OperationalError' in msg:
            raise sql.OperationalError(msg)

        if 'sqlite3.IntegrityError' in msg:
            raise sql.IntegrityError(msg)

        return ret
    
    def insert_into(self, table, data):
        ## Make an SQL-injection check for table and data!!
        try:
            # data must be a list of dictionaries
            self.validate_dict_raise(data)

            queries = []
            for row in data:
                columns = ", ".join(row.keys())    
                values = ", ".join(
                    f"'{v}'" if isinstance(v, str) and v!= "N/A" else "NULL"
                    for v in row.values()
                )
                query = f"INSERT INTO {table} ({columns}) VALUES({values});" 
                self.call_query(query)
        
        except TypeError as e:
            print(e)
        except sql.OperationalError as e:
            print(f"SQL insert into Failed due to operation error: {e}")
            raise

        except sql.IntegrityError as e:
            print(f"SQL insert into failed due to integrity error: {e}")
            raise

    def select(self, table, columns = None, where = None):
        try:
            query = "SELECT "
            
            if columns:
                query +=", ".join(columns)
            else:
                query += "*"
            
            query +=" FROM " + table

            if where:
                query +=" WHERE " + where;
            
            query += ";"
            return self.call_query(query)
            
        except TypeError as e:
            print(e)
        except sql.OperationalError as e:
            print(f"SQL Select Failed due to operation error: {e}")
            raise
        except sql.IntegrityError as e:
            print(f"SQL Select failed due to integrity error: {e}")
            raise

# Data Handling, Cleaning and Transformation
- Fetches raw data from DB
- Cleans missing values, standardizes formats
- Prepare for analysis: filtering, removing duplicates

## Data preparation
First, I analyze the data and check the following:
- Validate if the expected columns are present
- Validate data formats and their defined domains
- Handle any missing data
- Fill NaN values using a custom strategy
- inspect zero values if any

In [156]:
import pandas as pd
from urllib.parse import urlparse

BASE_URL = "https://www.investopedia.com/the-best-growth-stocks-8783031"

class StockDataHandler():
    SEARCH_KEYWORDS={'company', 'sector', 'stock', 'news', 'analysis', 'report', 'press', 'overview', 'financial', 
                     'guidance', 'result', 'outlook', 'market', 'trend', 'corporate', 'strategy', 'research'}
    
    NEWS_LINKS={"https://uk.investing.com/news/company-news", "https://www.stocktitan.net/news",
                "https://www.gurufocus.com/news", "https://www.insidermonkey.com/blog",
                "https://www.fool.co.uk/2025", "https://uk.investing.com/news/company-news",
                "https://www.investing.com/news/company-news", "https://simplywall.st/stocks/",
                "https://seekingalpha.com/article", "https://www.tradingview.com/symbols",
                "https://www.marketbeat.com/stocks", "https://newsable.asianetnews.com/markets",
                "https://www.stockstelegraph.com/"}
    
    stock_db = None
    debug = None
    def __init__(self, mock_db = False, debug = False):
        self.stock_db = StockAnalysisDB(mock_db=mock_db, debug=debug)
        self.debug = debug
        self.__ticker_keywords = set()
    
    # Remove not-needed legal endings
    def simplify_company_names(self, data_set):
        clean_data = set()
        for company in data_set:
            cleaned_company = company.replace(' Group Inc.','').replace(' Group, Inc.','')\
                                     .replace(' Holdings Inc.','').replace(' Holdings, Inc.','')\
                                     .replace(' Holdings Limited','').replace(' International Limited','')\
                                     .replace(' Holdings LLC','').replace(' International Limited','')\
                                     .replace(' S.A.','').replace(' Corp.','').replace(' Co.','')\
                                     .replace(', Inc.','').replace(' Inc.','').strip()
            clean_data.add(cleaned_company)
        return clean_data

    # Skip links that are not related to news, articles
    def is_link_might_be_useful(self, link):
        if not isinstance(link, str):
            return False
        
        useful = True
        link = link.lower()
        if any(kw in link for kw in ["javascript", "login", "signup", "advert", "privacy", "terms", "careers", "contact"]):
            useful = False
        return useful

    # We expect the stock_info table to be filled
    def get_data_as_keyword_from_db(self, columns=['ticker', 'company', 'sector']):
        
        data_df = self.stock_db.select("stock_info", columns).DataFrame()
        if data_df.empty:
            raise ValueError("Stock_info table is empty!")
        
        data_df = data_df.fillna("")
        # Sectors are too general, adding the text 'sector' will help with contex
        if 'sector' in data_df.columns:
            data_df['sector'] = data_df['sector'].apply(lambda s: s+ " sector")
        
        flattened_df = set(data_df.values.flatten())
        flattened_df.discard(" sector")                   
        flattened_df.discard("")
        return self.simplify_company_names(flattened_df)

    # At this state of the project the tickers won't change, so we can do this boosting
    def get_ticker_keywords(self):
        if len(self.__ticker_keywords) == 0:
            self.__ticker_keywords = self.get_data_as_keyword_from_db()
        return self.__ticker_keywords          

    
    
    # Return keyword otherwise None
    def get_keyword_if_link_contains(self, link):
        if not isinstance(link, str):
            return None

        keywords = self.get_ticker_keywords()
        keywords = {k.replace(' ', '-') for k in keywords} | {k.replace(' ', '_') for k in keywords}        

        pattern = r'(?<![a-zA-Z])(' + '|'.join(map(re.escape, keywords)) + r')(?=[^a-zA-Z]|$)'

        if self.debug:
            print(f"Regex Pattern: {pattern}")
            print(f"Testing link: {link}")
            print(f"Keywords: {keywords}")

        match = re.search(pattern, link, re.IGNORECASE)
        if match:
            return match.group(0).replace('-', ' ').replace('_', ' ') 

        return None   

# Web Spider
I will gather data from the Internet using a Webscraper:
- Tickers mentioned as good investment oportunities, whenever possible I will also save the company and sector
- I will save context from atricles about the company, and the sector

#### Ethical Considerations
...

## Webscraper
- Gather growth stocks from Investopedia
- Identify the tickers in the artices
- Store data gathered in a database


In [157]:
import re
import http.client
from urllib.request import urlopen
from urllib.request import Request
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin
from urllib.parse import urlparse
from furl import furl
from bs4 import BeautifulSoup
import time

def get_bs_from_url(url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
            "Referer": "https://www.google.com/", # simulate Google search
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1"
        }
        
        # To avoid 'Too many requests'
        time.sleep(5)
        
        response = urlopen(Request(url, headers=headers))
        return BeautifulSoup(response.read(), 'html.parser')

    except HTTPError as e:
        print(f"HTTP error: {e.code} - {e.reason}")
        return None
    
    except URLError as e:
        print(f"URL error for {url}: {e.reason}")
        return None

    except http.client.RemoteDisconnected:
        print("RemoteDisconnected: Remote host closed the connection.")
        return None

    except http.client.IncompleteRead as e:
        # e.partial contains whatever was successfully read
        html = e.partial
        print("Warning: IncompleteRead - partial response used.")
        return BeautifulSoup(html, 'html.parser')
    
    except ValueError  as e:
        print(f"ValueError for {url}: {e.reason}")
        return None


In [206]:
from collections import defaultdict
from collections import deque

class InvestopediaWebSpider:
    MAX_VISITED_LINKS = 1000
    
    debug = None
    links_to_explore = set()
    visited_links = set()
    keyword_link_context_map = defaultdict(lambda: defaultdict(set))

    stock_db = None
    data_handler = None

    def __init__(self, mock_db = False, debug = False):
        self.stock_db = StockAnalysisDB(mock_db=mock_db, debug=debug)
        self.data_handler = StockDataHandler(mock_db=mock_db, debug=debug)
        self.debug = debug
        self.bs = get_bs_from_url(BASE_URL)
        if not self.bs:
            raise ValueError("BeautifulSoup is not existing, the class can't be initiatied")

    ## Utility functions ##

    @staticmethod
    def convert_if_relative_URL(base_url, rel_url):
        if not isinstance(rel_url, str):
            print("URL is not a string")
            return None

        if not rel_url.isascii():
            print("URL is not ascii")
            return None

        # return None for javascript as it's not a link
        if rel_url.lower().startswith("javascript:"):
            print("URL javascript")
            return None

        # If empty, references the base url
        if rel_url == "":
            return base_url

        # Just return if not relative
        if urlparse(rel_url).scheme:
            return rel_url

        if not isinstance(base_url, str) or not base_url:
            print("base URL is not a string or empty")
            return None

        # Use scheme
        if rel_url.startswith("//"):
            base_scheme = urlparse(base_url).scheme
            if not base_scheme:
                print("Base URL has no scheme, cannot resolve `//` link.")
                return None
            return f"{base_scheme}:{rel_url}"

        if rel_url.startswith("#"):
            return urljoin(base_url, rel_url)
        
        parsed_base = urlparse(base_url)
        if parsed_base.path and not parsed_base.path.endswith('/'):
            base_url = parsed_base._replace(path=parsed_base.path + '/').geturl()

        return urljoin(base_url, rel_url)
                
    # Remove query params
    @staticmethod
    def normalize_url(base_link, url):
            
        # Convert references to full URLs
        # Returns none if not valid URL
        url = InvestopediaWebSpider.convert_if_relative_URL(base_link, url)
        if not url:
            return None

        f = furl(url)
        f.args = {}
        f.fragment = ''
        return f.url

    
    
    ## Scraper functions ##
    
    def scrape_ticker_data_from_investopedia(self):
        # Look for the table with the world 'stock'
        table = None
        for tbl in self.bs.find_all("table"):
            if tbl.find("th", string=lambda s: s and "stock" in s.lower()):
                table = tbl
                break

        if table:
            # First row of the table body contains the indexes
            first_row = table.find("tbody").find("tr")
            column_name_indexes = {}
            for i, td in enumerate(first_row.find_all("td", recurisive=False)):
                text = td.get_text(strip=True).lower()
                if "ticker" in text:
                    column_name_indexes["ticker"] = i
                elif "company" in text:
                    column_name_indexes["company"] = i
                elif "sector" in text:
                    column_name_indexes["sector"] = i
                i+=1

            # Get next data from table
            all_ticker_data = []
            for row in first_row.find_next_siblings("tr"):
                ticker_data = {}
                cells = row.find_all("td")
                
                # Ticker has a link to the profile
                link = cells[column_name_indexes["ticker"]].find('a')
                if link:
                    ticker_data["investopedia_link"] = link.attrs['href']
                    self.links_to_explore.add(link.attrs['href'])
                
                # Extract data from each cell
                ticker_data["ticker"] = cells[column_name_indexes["ticker"]].get_text(strip=True)
                ticker_data["company"] = cells[column_name_indexes["company"]].get_text(strip=True)
                ticker_data["sector"] = cells[column_name_indexes["sector"]].get_text(strip=True)

                all_ticker_data.append(ticker_data)

            self.stock_db.insert_into("stock_info", all_ticker_data)
    
    # Scrapes the BeautifulSoup for any occurrence of tickers, companies and sector.
    # If a keyword is found in a header, it also collects the next paragraph.
    # Stores the context in the database.
    def scrape_context_for_keywords(self, bs, link): 
        keywords = self.data_handler.get_ticker_keywords()        
        elements = bs.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p'])
        for i, e in enumerate(elements):
            text = e.get_text(separator=" ", strip=True)
            for keyword in keywords:

                # test with regex if it's a match (important for tickers as they are very small)
                if re.search(r'\b' + re.escape(keyword) + r'\b', text, re.IGNORECASE):    
                    
                    # If found it, gather all text from siblings and under
                    text_from_siblings_and_under = ""
                    for sibling in e.find_next_siblings():
                        text_from_siblings_and_under += sibling.get_text(separator=" ", strip=True)
                    limited_text = (text + " " + text_from_siblings_and_under)[:500]
                    self.keyword_link_context_map[keyword][link].add(limited_text)
    
    def collect_context_with_spider_BFS(self, max_depth = 5):
        if self.debug:
                print(f"{self.__class__.__name__}.{inspect.currentframe().f_code.co_name}")

        # Use FIFO for BFS
        queue = deque(self.data_handler.NEWS_LINKS)
        
        depth = 0
        while queue and depth < max_depth:
            print(f"\n--- depth = {depth} ---\n")
            depth += 1

            level_size = len(queue)
            for _ in range(level_size):
                if not queue: # or len(self.visited_links) < self.MAX_VISITED_LINKS
                    break
            
                current_link = queue.popleft()            
                if current_link in self.visited_links:
                    continue

                print(f"Processing {current_link}...")
                bs = get_bs_from_url(current_link)
                if not bs:
                    if self.debug:
                        print("bs is None")
                    continue

                self.visited_links.add(current_link)
                self.scrape_context_for_keywords(bs, current_link)

                for link in bs.find_all('a', limit = 50):
                    if 'href' in link.attrs:
                        link_text = self.normalize_url(current_link, link['href'])
                        if link_text and link_text not in self.visited_links:
                            queue.append(link_text)

In [207]:
ws = InvestopediaWebSpider()

In [ ]:
ws.collect_context_with_spider_BFS()


--- depth = 0 ---

Processing https://www.fool.co.uk/2025...
Processing https://www.gurufocus.com/news...
Processing https://www.investing.com/news/company-news...
Processing https://simplywall.st/stocks/...
Processing https://www.stocktitan.net/news...
HTTP error: 404 - Not Found
Processing https://www.insidermonkey.com/blog...
Processing https://seekingalpha.com/article...
HTTP error: 403 - Forbidden
Processing https://www.marketbeat.com/stocks...
Processing https://www.tradingview.com/symbols...
Processing https://uk.investing.com/news/company-news...
Processing https://www.stockstelegraph.com/...
Processing https://newsable.asianetnews.com/markets...

--- depth = 1 ---

Processing https://www.fool.co.uk/account...
Processing https://www.fool.co.uk/premium-hub...
Processing https://www.fool.co.uk/wp-login.php...
Processing https://www.fool.co.uk/...
Processing https://www.fool.co.uk/category/investing/...
Processing https://www.fool.co.uk/tickers/...
Processing https://www.fool.co.

In [ ]:
for kw in ws.keyword_link_context_map:
    print(kw)

In [15]:
# %sql sqlite:///stockAnalysis.db "SELECT * FROM stock_info;"

In [16]:
# %sql sqlite:///stockAnalysis.db "DELETE FROM stock_info;"

In [17]:
# %sql sqlite:///stockAnalysis.db "SELECT * FROM company_context;"

In [18]:
# %sql sqlite:///stockAnalysis.db "SELECT * FROM sector_context;"

# Optional: DataPipeline
If I have time and want to automate

# Data Analysis
- uses cleaned data from DB
- applies statistics, visualization, ML
- final output: insights

# Unit Tests

### Create mock tables in memory database

In [19]:
# To avoid conflict with different versions
%config SqlMagic.style = 'Default'
%reload_ext sql

In [20]:
%%sql sqlite:///:memory:
CREATE TABLE IF NOT EXISTS stock_info(
    ticker VARCHAR(10) NOT NULL UNIQUE,
    company TEXT,
    sector VARCHAR(100),
    investopedia_link TEXT,
    PRIMARY KEY (ticker),
    FOREIGN KEY (company) REFERENCES company_context(company),
    FOREIGN KEY (sector) REFERENCES sector_context(sector)
);

Done.


[]

In [21]:
%%sql sqlite:///:memory:
CREATE TABLE IF NOT EXISTS company_context(
    company TEXT NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (company)
);

Done.


[]

In [22]:
%%sql sqlite:///:memory:
CREATE TABLE  IF NOT EXISTS sector_context(
    sector VARCHAR(100) NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (sector)
);

Done.


[]

## Test Database class

In [23]:
%sql sqlite:///:memory:

import unittest
from io import StringIO
from unittest.mock import patch
import sqlite3 as sql

class TestDatabase(unittest.TestCase):

    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

        # Create mock DB
        self.db = StockAnalysisDB(mock_db=True)

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()

    # Define test type when we expect no exception
    def assertNotRaises(self, exception, func, *args, **kwargs):
        try:
            func(*args, **kwargs)
        except exception as e:
            self.fail(f"Expected no {exception}, but got: {e}")

    def test_db_class_exists(self):
        self.assertIsNotNone(StockAnalysisDB)

    
     ## call_query function ##

    def test_call_query_exists(self):
        self.assertIsNotNone(StockAnalysisDB.call_query)
        
    @patch("__main__.get_ipython")
    def test_call_query_handles_exeptions(self, mock_ipython):

        # Mock an OperationalError
        mock_ipython.return_value.run_line_magic.side_effect = sql.OperationalError("sqlite3.OperationalError: syntax error")
        with self.assertRaises(sql.OperationalError) as context:
            self.db.call_query("INVALID SQL")
        self.assertIn("sqlite3.OperationalError", str(context.exception))

        # Mock an IntegrityError
        mock_ipython.return_value.run_line_magic.side_effect = sql.IntegrityError("sqlite3.IntegrityError: unique constraint failed")
        with self.assertRaises(sql.IntegrityError) as context:
            self.db.call_query("INSERT INTO stock_info VALUES ('XXX')")
        self.assertIn("sqlite3.IntegrityError", str(context.exception))
            
    
    ## validate_dict_raise (insert_into validator) function ##

    def test_validate_dict_raise_exists(self):
        self.assertIsNotNone(StockAnalysisDB.validate_dict_raise)

    def test_validate_dict_raise_exception_empty_1(self):
        test_data = None
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_2(self):
        test_data = ""
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_3(self):
        test_data = []
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_4(self):
        test_data = {}
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_not_list(self):
        test_dict = {'col1': 'link',
                        'col2': 'XXX',
                        'col3': 'text',
                        'col4': 'text'}
        with self.assertRaises(TypeError):
             StockAnalysisDB.validate_dict_raise(test_dict)

    def test_validate_dict_raise_exception_not_dict_in_list(self):
        test_list = ['text1', 'text2', 'text3', 'text4']
        with self.assertRaises(TypeError):
            StockAnalysisDB.validate_dict_raise(test_list)

    def test_validate_dict_not_raise_exception(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        self.assertNotRaises(TypeError, self.db.validate_dict_raise, my_test_list_dict) 

    
    ## insert_into function ##
    
    def test_insert_into_exists(self):
        self.assertIsNotNone(StockAnalysisDB.insert_into)

    def test_insert_into_wrong_table(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        with self.assertRaisesRegex(sql.OperationalError, r"syntax error"):
            self.db.insert_into("table", my_test_list_dict)

    def test_insert_into_wrong_column_name(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        with self.assertRaisesRegex(sql.OperationalError, r"has no column named"):
            self.db.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_no_primary_key(self):
        my_test_list_dict = [{'investopedia_link': 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=TEM',
                              'company': 'Tempus AI Inc.',
                              'sector': 'Health Care'},]
        with self.assertRaisesRegex(sql.IntegrityError, r"constraint failed"):
            self.db.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_wrong_key_type(self):
        my_test_list_dict = [{'investopedia_link': 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=TEM',
                              'ticker': 3,
                              'company': 'Tempus AI Inc.',
                              'sector': 'Health Care'},]
        with self.assertRaisesRegex(sql.IntegrityError, r"constraint failed"):
            self.db.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_correct(self):
        my_test_list_dict = [{'investopedia_link': 'link',
                              'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector'}]
        
        self.db.insert_into("stock_info", my_test_list_dict)
        res = self.db.call_query("SELECT investopedia_link,ticker,company,sector FROM stock_info WHERE ticker = 'XXX'")

        # Verify and delete data
        self.assertEqual(res[0][0], 'link')
        self.assertEqual(res[0][1], 'XXX')
        self.assertEqual(res[0][2], 'company')
        self.assertEqual(res[0][3], 'sector')

        self.db.call_query("DELETE FROM stock_info WHERE ticker='XXX'")

    
    ## select function ##
    
    def test_select_exists(self):
        self.assertIsNotNone(StockAnalysisDB.select)

    @patch("__main__.StockAnalysisDB.call_query")
    def test_select_query_with_no_args(self, mock_execute):
        self.db.select("table")
        
        # Check if query sting is correct
        mock_execute.assert_called_once_with("SELECT * FROM table;")

    @patch("__main__.StockAnalysisDB.call_query")
    def test_select_query_with_columns(self, mock_execute):
        self.db.select("table", ['col1', 'col2', 'col3'])
        
        # Check if query sting is correct
        mock_execute.assert_called_once_with("SELECT col1, col2, col3 FROM table;")

    def test_select_return_with_no_args(self):
        # Fill the mock db
        my_test_list_dict = [{'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector',
                              'investopedia_link': 'link'}]
        self.db.insert_into("stock_info", my_test_list_dict)

        # Verify and delete data
        res = self.db.select("stock_info")
        self.assertEqual(len(res), 1)
        self.assertEqual(len(res[0]), 4)
        
        self.assertEqual(res[0][0], 'XXX')
        self.assertEqual(res[0][1], 'company')
        self.assertEqual(res[0][2], 'sector')
        self.assertEqual(res[0][3], 'link')

        self.db.call_query("DELETE FROM stock_info WHERE ticker='XXX'")

    def test_select_return_with_col(self):
        # Fill the mock db
        my_test_list_dict = [{'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector',
                              'investopedia_link': 'link'},
                              {'ticker': 'YYY',
                              'company': 'company',
                              'sector': 'sector',
                              'investopedia_link': 'link'}]
        self.db.insert_into("stock_info", my_test_list_dict)

        # Verify and delete data
        res = self.db.select("stock_info", ['ticker'])
        self.assertEqual(len(res), 2)
        self.assertEqual(len(res[0]), 1)
        
        self.assertEqual(res[0][0], 'XXX')
        self.assertEqual(res[1][0], 'YYY')

        self.db.call_query("DELETE FROM stock_info WHERE ticker IN ('XXX', 'YYY')")

    def test_select_return_with_cols(self):        
        # Fill the mock db
        my_test_list_dict = [{'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector',
                              'investopedia_link': 'link'},
                              {'ticker': 'YYY',
                              'company': 'company2',
                              'sector': 'sector',
                              'investopedia_link': 'link'}]
        self.db.insert_into("stock_info", my_test_list_dict)

        # Verify and delete data
        res = self.db.select("stock_info", ['ticker', 'company'])
        self.assertEqual(len(res), 2)
        self.assertEqual(len(res[0]), 2)
        
        self.assertEqual(res[0][0], 'XXX')
        self.assertEqual(res[0][1], 'company')

        self.assertEqual(res[1][0], 'YYY')
        self.assertEqual(res[1][1], 'company2')

        self.db.call_query("DELETE FROM stock_info WHERE ticker IN ('XXX', 'YYY')")

suite = unittest.TestLoader().loadTestsFromTestCase(TestDatabase)
unittest.TextTestRunner(verbosity=2).run(suite)

test_call_query_exists (__main__.TestDatabase.test_call_query_exists) ... ok
test_call_query_handles_exeptions (__main__.TestDatabase.test_call_query_handles_exeptions) ... ok
test_db_class_exists (__main__.TestDatabase.test_db_class_exists) ... ok
test_insert_into_correct (__main__.TestDatabase.test_insert_into_correct) ... ok
test_insert_into_exists (__main__.TestDatabase.test_insert_into_exists) ... ok
test_insert_into_no_primary_key (__main__.TestDatabase.test_insert_into_no_primary_key) ... ok
test_insert_into_wrong_column_name (__main__.TestDatabase.test_insert_into_wrong_column_name) ... ok
test_insert_into_wrong_key_type (__main__.TestDatabase.test_insert_into_wrong_key_type) ... ok
test_insert_into_wrong_table (__main__.TestDatabase.test_insert_into_wrong_table) ... ok
test_select_exists (__main__.TestDatabase.test_select_exists) ... ok
test_select_query_with_columns (__main__.TestDatabase.test_select_query_with_columns) ... ok
test_select_query_with_no_args (__main__.TestData

<unittest.runner.TextTestResult run=23 errors=0 failures=0>

In [24]:
# res = %sql sqlite:///:memory: "SELECT * FROM stock_info;"

## Test Data Handler

In [25]:
%sql sqlite:///:memory:
import inspect

class TestStockDataHandler(unittest.TestCase):

    def setUp(self):
        self.data_handler = StockDataHandler(mock_db=True)
        
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()

    def test_StockDataHandler_exists(self):
        self.assertIsNotNone(StockDataHandler)


    ## simplify_company_names function ##

    def test_simplify_company_names_exists(self):
        self.assertIsNotNone(StockDataHandler.simplify_company_names)
    
    def test_simplify_company_names_removes_suffixes(self):
        test_data = {"A Group Inc.","B Group, Inc.", "C Holdings Inc.", "D Holdings, Inc.", "E Holdings Limited",
                     "F International Limited", "G Holdings LLC", "H International Limited", "I S.A.", "J Corp.",
                     "K Co.", "L Inc.", "M, Inc."}
        res = self.data_handler.simplify_company_names(test_data)
        self.assertSetEqual(res, {"A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M"})

    # if suffixes are changed, the test has to be changed too!
    def test_simplify_company_names_suffix_changed(self):
        code = inspect.getsource(self.data_handler.simplify_company_names)

        # find quoted strings
        suffixes_from_code = set(re.findall(r"\.replace\(\s*'([^']+?)'\s*,\s*''\s*\)", code))
        expected_suffixes = {" Group Inc."," Group, Inc.", " Holdings Inc.", " Holdings, Inc.", " Holdings Limited",
                     " International Limited", " Holdings LLC", " International Limited", " S.A.", " Corp.",
                     " Co.", " Inc.", ", Inc."}

        self.assertEqual(suffixes_from_code, expected_suffixes, 
                         f"New suffixes were added: {suffixes_from_code - expected_suffixes}\n"
                         f"Expected suffixes missing: {expected_suffixes - suffixes_from_code}\n"
                         "Update the test case with the correct suffixes!")


    ## is_link_might_be_useful function ##

    def test_is_link_might_be_useful_exists(self):
        self.assertIsNotNone(StockDataHandler.is_link_might_be_useful)

    def test_is_link_might_be_useful_not_string(self):
        self.assertFalse(self.data_handler.is_link_might_be_useful(234))
        self.assertFalse(self.data_handler.is_link_might_be_useful([]))
        self.assertFalse(self.data_handler.is_link_might_be_useful({}))

    def test_is_link_might_be_useful_1(self):
        self.assertFalse(self.data_handler.is_link_might_be_useful("link_with_javascript_salong"))

    def test_is_link_might_be_useful_2(self):
        self.assertFalse(self.data_handler.is_link_might_be_useful("link_with_terms"))

    def test_is_link_might_be_useful_3(self):
        self.assertTrue(self.data_handler.is_link_might_be_useful("link_without_the_keywords"))
    
    def test_is_link_might_be_useful_suffix_changed(self):
        # if suffixes are changed, the test has to be changed too!
        code = inspect.getsource(self.data_handler.is_link_might_be_useful)

        # find quoted strings
        suffixes_from_code = re.findall(r'"([^"]+)"', code)
        expected_suffixes = ["javascript", "login", "signup", "advert", "privacy", "terms", "careers", "contact"]

        self.assertEqual(suffixes_from_code, expected_suffixes, "Update the test case with the correct suffixes!")


    ## get_data_as_keyword_from_db function ##

    def test_get_data_as_keyword_from_db_exists(self):
        self.assertIsNotNone(StockDataHandler.get_data_as_keyword_from_db)

    def test_get_data_as_keyword_from_db_DB_empty(self):
        # Make sure it's empty
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")
        with self.assertRaises(ValueError):
            self.data_handler.get_data_as_keyword_from_db()

    def test_get_data_as_keyword_from_db_no_args(self):
        
        # Make sure SB is empty, and fill
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")
        spider = InvestopediaWebSpider(mock_db=True)
        spider.scrape_ticker_data_from_investopedia()
        
        ret = self.data_handler.get_data_as_keyword_from_db()
        self.assertEqual(ret, {'CYD', 'China Yuchai', 'Finance sector', 'GDS', 'GRRR', 'Gorilla Technology', 'Health Technology sector', 'INOD',
         'Innodata', 'ORGO', 'Organogenesis', 'PRCH', 'Porch', 'Producer Manufacturing sector', 'ROOT', 'RUA', 'Root',
         'Russell 3000', 'SLDB', 'SWTX', 'Solid Biosciences', 'SpringWorks Therapeutics', 'Technology Services sector', 'VNET'})

        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")


    ## get_keyword_if_link_contains function ##

    def test_get_keyword_if_link_contains_not_string(self):
        self.assertIsNone(self.data_handler.get_keyword_if_link_contains(123))
        self.assertIsNone(self.data_handler.get_keyword_if_link_contains([]))
        self.assertIsNone(self.data_handler.get_keyword_if_link_contains({}))

    def test_get_keyword_if_link_contains_ticker(self):
        
        # Make sure SB is empty, and fill
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")
        spider = InvestopediaWebSpider(mock_db=True)
        spider.scrape_ticker_data_from_investopedia()
        
        self.assertEqual(self.data_handler.get_keyword_if_link_contains("link-with-GRRR-ticker"), 'GRRR')
        self.assertEqual(self.data_handler.get_keyword_if_link_contains("GRRR-ticker"), 'GRRR')
        self.assertEqual(self.data_handler.get_keyword_if_link_contains("GRRR.html"), 'GRRR')
        self.assertEqual(self.data_handler.get_keyword_if_link_contains("something-GRRR.html"), 'GRRR')

        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")

    def test_get_keyword_if_link_contains_spaces_1(self):
        
        # Make sure SB is empty, and fill
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")
        spider = InvestopediaWebSpider(mock_db=True)
        spider.scrape_ticker_data_from_investopedia()
        
        self.assertEqual(self.data_handler.
                         get_keyword_if_link_contains("something-Health-Technology-sector.html"), 'Health Technology sector')
        self.assertEqual(self.data_handler.
                         get_keyword_if_link_contains("Health-Technology-sector.html"), 'Health Technology sector')
        self.assertEqual(self.data_handler.
                         get_keyword_if_link_contains("/Health-Technology-sector/valami.html"), 'Health Technology sector')

        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")

    def test_get_keyword_if_link_contains_spaces_2(self):
        
        # Make sure SB is empty, and fill
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")
        spider = InvestopediaWebSpider(mock_db=True)
        spider.scrape_ticker_data_from_investopedia()
        
        self.assertEqual(self.data_handler.
                         get_keyword_if_link_contains("something_Health_Technology_sector.html"), 'Health Technology sector')
        self.assertEqual(self.data_handler.
                         get_keyword_if_link_contains("Health_Technology_sector_something.html"), 'Health Technology sector')
        self.assertEqual(self.data_handler.
                         get_keyword_if_link_contains("/Health_Technology_sector/something"), 'Health Technology sector')

        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")

    def test_get_keyword_if_link_contains_spaces_3(self):
        
        # Make sure SB is empty, and fill
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")
        spider = InvestopediaWebSpider(mock_db=True)
        spider.scrape_ticker_data_from_investopedia()
        
        
        self.data_handler.stock_db.call_query("DELETE FROM stock_info;")

    


suite = unittest.TestLoader().loadTestsFromTestCase(TestStockDataHandler)
unittest.TextTestRunner(verbosity=2).run(suite)

test_StockDataHandler_exists (__main__.TestStockDataHandler.test_StockDataHandler_exists) ... ok
test_get_data_as_keyword_from_db_DB_empty (__main__.TestStockDataHandler.test_get_data_as_keyword_from_db_DB_empty) ... ok
test_get_data_as_keyword_from_db_exists (__main__.TestStockDataHandler.test_get_data_as_keyword_from_db_exists) ... ok
test_get_data_as_keyword_from_db_no_args (__main__.TestStockDataHandler.test_get_data_as_keyword_from_db_no_args) ... ok
test_get_keyword_if_link_contains_not_string (__main__.TestStockDataHandler.test_get_keyword_if_link_contains_not_string) ... ok
test_get_keyword_if_link_contains_spaces_1 (__main__.TestStockDataHandler.test_get_keyword_if_link_contains_spaces_1) ... ok
test_get_keyword_if_link_contains_spaces_2 (__main__.TestStockDataHandler.test_get_keyword_if_link_contains_spaces_2) ... ok
test_get_keyword_if_link_contains_spaces_3 (__main__.TestStockDataHandler.test_get_keyword_if_link_contains_spaces_3) ... ok
test_get_keyword_if_link_contains_ti

<unittest.runner.TextTestResult run=18 errors=0 failures=0>

In [26]:
# spider = InvestopediaWebSpider(mock_db=True, debug=True)
# spider.scrape_ticker_data_from_investopedia()

In [27]:
# res = %sql sqlite:///:memory: "DELETE FROM stock_info;"

## Test Web Spider

### Test BeautifulSoup function

In [28]:
%sql sqlite:///:memory:

class TestGetBSFromURL(unittest.TestCase):

    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()
    
    # test what happens when bs can't be created by mocking the urlopen function
    @patch("__main__.urlopen")
    def test_http_error(self, mock_urlopen):
        mock_urlopen.side_effect = HTTPError(
            url = BASE_URL,
            code = 404,
            msg = 'Not Found',
            hdrs = {},
            fp = None)

        bs = get_bs_from_url(BASE_URL)
        self.assertIsNone(bs)

    def test_valid_url(self):
        bs = get_bs_from_url(BASE_URL)
        self.assertIsNotNone(bs)
        self.assertIn("Growth Stocks", str(bs))
    
    # test what happens when the link is wrong by mocking the urlopen function
    @patch("__main__.urlopen")
    def test_wrong_url_and_no_bs(self, mock_urlopen):
        mock_urlopen.side_effect = URLError("No route to host")
        bs = get_bs_from_url("https://www.invesftopedia.com/the-best-growth-stocks-8783031")
        self.assertIsNone(bs)

suite = unittest.TestLoader().loadTestsFromTestCase(TestGetBSFromURL)
unittest.TextTestRunner(verbosity=2).run(suite)

test_http_error (__main__.TestGetBSFromURL.test_http_error) ... ok
test_valid_url (__main__.TestGetBSFromURL.test_valid_url) ... ok
test_wrong_url_and_no_bs (__main__.TestGetBSFromURL.test_wrong_url_and_no_bs) ... ok

----------------------------------------------------------------------
Ran 3 tests in 15.656s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

In [ ]:
class TestWebSpider(unittest.TestCase):
    def setUp(self):
        self.ws = InvestopediaWebSpider(mock_db=True)
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()
        pass
    
    # def test_webspider_class_exists(self):
    #     self.assertIsNotNone(InvestopediaWebSpider)

    
    # Utility functions #
    ## convert_relative_URL function ##

    # def test_convert_relative_URL_exists(self):
    #     self.assertIsNotNone(InvestopediaWebSpider.convert_if_relative_URL)
    
    # def test_convert_relative_URL_non_string(self):
    #     self.assertIsNone(self.ws.convert_if_relative_URL("",34))
    #     self.assertIsNone(self.ws.convert_if_relative_URL("",[]))
    #     self.assertIsNone(self.ws.convert_if_relative_URL("",{}))

    # def test_convert_relative_URL_base_empty(self):
    #     self.assertIsNone(self.ws.convert_if_relative_URL("","#something"))
    
    # def test_convert_relative_URL_base_non_string(self):
    #     self.assertIsNone(self.ws.convert_if_relative_URL(34,"#something"))
    #     self.assertIsNone(self.ws.convert_if_relative_URL([],"#something"))
    #     self.assertIsNone(self.ws.convert_if_relative_URL({},"#something"))

    # def test_convert_relative_URL_non_relative(self):
    #     self.assertEqual("http://something.com", self.ws.convert_if_relative_URL("","http://something.com"))
    #     self.assertEqual("https://something.com", self.ws.convert_if_relative_URL("","https://something.com"))
    #     self.assertEqual("https://something.com/path/to/smg", self.ws.convert_if_relative_URL("","https://something.com/path/to/smg"))
    #     self.assertEqual("http://something.com/search?q=query", self.ws.convert_if_relative_URL("","http://something.com/search?q=query"))
    #     self.assertEqual("http://something.com/page#section1", self.ws.convert_if_relative_URL("","http://something.com/page#section1"))
    #     self.assertEqual("http://something.com/path?q=query#section1", self.ws.convert_if_relative_URL("","http://something.com/path?q=query#section1"))

    # def test_convert_relative_URL_converted_1(self):
    #     self.assertEqual("http://something.com/path/to/something", self.ws.convert_if_relative_URL("http://something.com","/path/to/something"))

    # def test_convert_relative_URL_converted_2(self):
    #     self.assertEqual("http://something.com/to/something", self.ws.convert_if_relative_URL("http://something.com/path","/to/something"))
    
    # def test_convert_relative_URL_converted_3(self):
    #     self.assertEqual("http://something.com/something", self.ws.convert_if_relative_URL("http://something.com","./something"))
 
    # def test_convert_relative_URL_converted_4(self):
    #     self.assertEqual("http://something.com/path/something", self.ws.convert_if_relative_URL("http://something.com/path","./something"))

    # def test_convert_relative_URL_converted_5(self):
    #     self.assertEqual("http://something.com/path/something", self.ws.convert_if_relative_URL("http://something.com/path/to","../something"))

    # def test_convert_relative_URL_converted_6(self):
    #     self.assertEqual("http://something.com/path/to/something", self.ws.convert_if_relative_URL("http://something.com/path/to","something"))

    # def test_convert_relative_URL_converted_7(self):
    #     self.assertEqual("http://something.com/search?q=query", self.ws.convert_if_relative_URL("http://something.com","/search?q=query"))

    # def test_convert_relative_URL_converted_8(self):
    #     self.assertEqual("http://something.com/page#section", self.ws.convert_if_relative_URL("http://something.com/page","#section"))

    # def test_convert_relative_URL_converted_9(self):
    #     self.assertEqual("http://something.com/path?q=query#section", self.ws.convert_if_relative_URL("http://something.com","/path?q=query#section"))

    # # inherits schema
    # def test_convert_relative_URL_converted_10(self):
    #     self.assertEqual("http://example.com/path", self.ws.convert_if_relative_URL("http://something.com", "//example.com/path"))
    #     self.assertEqual("https://example.com/path", self.ws.convert_if_relative_URL("https://something.com", "//example.com/path"))
        
    # def test_convert_relative_URL_empty(self):
    #     self.assertEqual("http://something.com/path", self.ws.convert_if_relative_URL("http://something.com/path", ""))


    ## normalize_url function ##

    def test_normalize_url_exists(self):
        self.assertIsNotNone(InvestopediaWebSpider.normalize_url)

    def test_normalize_url_no_query(self):
        self.assertEqual("http://example.com/path", self.ws.normalize_url("", "http://example.com/path"))
        self.assertEqual("https://example.com", self.ws.normalize_url("", "https://example.com"))
        self.assertEqual("https://example.com/", self.ws.normalize_url("", "https://example.com/"))
    
    def test_normalize_url_with_query_and_fragment(self):
        self.assertEqual("https://example.com/path/to/page", 
                         self.ws.normalize_url("", "https://example.com/path/to/page?query=123#section"))
    
    def test_normalize_url_with_complex_path(self):
        self.assertEqual("https://www.example.com/path/to/page.html",
                         self.ws.normalize_url("", "https://www.example.com/path/to/page.html?foo=bar&baz=qux#anchor"))

    def test_normalize_url_relative(self):
        self.assertEqual("https://www.example.com/path/to/page.html",
                         self.ws.normalize_url("https://www.example.com", "/path/to/page.html?foo=bar&baz=qux#anchor"))

    # ## scrape_ticker_data_from_investopedia function ##
    
    # def test_scrape_ticker_data_from_investopediaÜexists(self):
    #     self.assertIsNotNone(InvestopediaWebSpider.scrape_ticker_data_from_investopedia)

    # def test_scrape_ticker_data_from_investopedia_data_fills_DB(self):
    #     test_spider = InvestopediaWebSpider(mock_db=True)
    #     test_spider.scrape_ticker_data_from_investopedia()

    #     # DB should be filled with expected results
    #     res = %sql "SELECT * FROM stock_info"
    #     expected_list = [('GRRR','Gorilla Technology Group Inc.', 'Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=GRRR'),
    #     ('ORGO','Organogenesis Holdings Inc.', 'Health Technology','https://www.investopedia.com/markets/quote?tvwidgetsymbol=ORGO'),
    #     ('CYD','China Yuchai International Limited','Producer Manufacturing','https://www.investopedia.com/markets/quote?tvwidgetsymbol=CYD'),
    #     ('SLDB','Solid Biosciences Inc.','Health Technology', 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=SLDB'),
    #     ('GDS', 'GDS Holdings Limited', 'Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=GDS'),
    #     ('PRCH','Porch Group, Inc.','Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=PRCH'),
    #     ('ROOT','Root, Inc.','Finance','https://www.investopedia.com/markets/quote?tvwidgetsymbol=ROOT'),
    #     ('SWTX','SpringWorks Therapeutics, Inc.','Health Technology','https://www.investopedia.com/markets/quote?tvwidgetsymbol=SWTX'),
    #     ('VNET','VNET Group, Inc.','Technology Services', 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=VNET'),
    #     ('INOD','Innodata Inc.','Technology Services','https://www.investopedia.com/markets/quote?tvwidgetsymbol=INOD'),
    #     ('RUA',	'Russell 3000',	None,'https://www.investopedia.com/markets/quote?tvwidgetsymbol=RUA')]

    #     %sql "DELETE FROM stock_info;"

    #     self.assertListEqual(res, expected_list)

        
    # ## collect_links_from_self_bs function ##
    
    # def test_collect_links_exists(self):
    #     self.assertIsNotNone(InvestopediaWebSpider.collect_links_from_self_bs)

suite = unittest.TestLoader().loadTestsFromTestCase(TestWebSpider)
unittest.TextTestRunner(verbosity=2).run(suite)

test_normalize_url_exists (__main__.TestWebSpider.test_normalize_url_exists) ... ok
test_normalize_url_no_query (__main__.TestWebSpider.test_normalize_url_no_query) ... ok
test_normalize_url_relative (__main__.TestWebSpider.test_normalize_url_relative) ... 

## Analysis

## Conclusions
- 

## Possible improvements for the project
1. Work on SQL-inject
2. SChedule with Pipeline
3. 

## Difficulties with data
scraper ahh a lot

## References
$^1$: https://www.investopedia.com/terms/g/growthstock.asp

In [15]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
